In [4]:
import os
import glob
import numpy as np
from scipy.io import wavfile
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

folders = ['../Data/speech_commands/on', '../Data/speech_commands/off']

X = []
y = []

print("در حال استخراج ویژگی‌های زمان‌دار (تکنیک Windowing)...")

for label in folders:
    if not os.path.exists(label):
        continue
        
    wav_files = glob.glob(os.path.join(label, '*.wav'))
    for filepath in wav_files:
        try:
            sample_rate, data = wavfile.read(filepath)
            
            if len(data.shape) > 1:
                data = data[:, 0]
            if len(data) == 0:
                continue
                
            data = data.astype(int)
            
            # === تکنیک جدید: تقسیم صدا به 4 بخش مساوی ===
            # این کار به مدل درک زمان‌بندی (ابتدا و انتهای کلمه) می‌دهد
            chunks = np.array_split(data, 4)
            features = []
            
            for chunk in chunks:
                if len(chunk) == 0:
                    features.extend([0, 0])
                    continue
                
                # محاسبه انرژی و ZCR برای همین بخش کوچک
                energy = int(np.sum(np.abs(chunk)) / len(chunk))
                signs = chunk > 0
                zcr = int(np.sum(signs[1:] != signs[:-1]))
                
                # اضافه کردن به لیست ویژگی‌های این فایل
                features.extend([energy, zcr])
                
            # اگر 4 بخش داشتیم، 8 ویژگی به دست می‌آید
            if len(features) == 8:
                X.append(features)
                y.append(label)
        except Exception as e:
            pass

X = np.array(X)
y = np.array(y)

print(f"استخراج تمام شد! تعداد نمونه‌ها: {len(X)}")

# 2. آموزش مدل
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# عمق درخت را 7 می‌گذاریم تا الگوهای 8 ویژگی را بهتر یاد بگیرد
# این عمق برای رم 192 کیلوبایتی میکروکنترلر شما به شدت سبک و امن است
clf = DecisionTreeClassifier(max_depth=7, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"دقت مدل جدید روی داده‌های تست: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# 3. استخراج مدل برای میکروپایتون
tree = clf.tree_
features = tree.feature.tolist()
thresholds = [round(x, 2) for x in tree.threshold.tolist()]
left = tree.children_left.tolist()
right = tree.children_right.tolist()

node_classes = [str(clf.classes_[np.argmax(v)]) for v in tree.value]
unique_classes = list(set(node_classes))
class_indices = [unique_classes.index(c) for c in node_classes]

with open("speech_model.py", "w", encoding="utf-8") as f:
    f.write(f"features = {features}\n")
    f.write(f"thresholds = {thresholds}\n")
    f.write(f"left = {left}\n")
    f.write(f"right = {right}\n")
    f.write(f"unique_classes = {unique_classes}\n")
    f.write(f"class_indices = {class_indices}\n")

print("فایل speech_model.py ساخته شد! آماده برای انتقال به برد.")

در حال استخراج ویژگی‌های زمان‌دار (تکنیک Windowing)...
استخراج تمام شد! تعداد نمونه‌ها: 4706
دقت مدل جدید روی داده‌های تست: 78.98%
فایل speech_model.py ساخته شد! آماده برای انتقال به برد.


In [5]:
accuracy_score(y_test, y_pred) * 100

78.98089171974523

In [ ]:
import gc
import time
import array
import os
import speech_model  # فایل مدلی که قبلاً ساختیم

gc.collect()

def predict_speech(features_list):
    node = 0
    while speech_model.left[node] != -1:
        feature_idx = speech_model.features[node]
        if features_list[feature_idx] <= speech_model.thresholds[node]:
            node = speech_model.left[node]
        else:
            node = speech_model.right[node]
            
    idx = speech_model.class_indices[node]
    raw_label = speech_model.unique_classes[idx]
    
    # مرتب‌سازی نام کلاس خروجی
    clean_label = raw_label.split('/')[-1].split('\\')[-1].upper()
    return clean_label

def process_wav_file(filename):
    print(f"Opening '{filename}' for processing...")
    
    # در فایل‌های 16 کیلوهرتز 1 ثانیه‌ای، هر تکه (0.25 ثانیه) برابر با 4000 نمونه است
    # چون هر نمونه 16 بیتی (2 بایت) است، پس هر تکه 8000 بایت حجم دارد
    CHUNK_BYTES = 8000 
    features = []
    
    try:
        # فایل را به صورت باینری (rb) باز می‌کنیم
        with open(filename, 'rb') as f:
            # رد کردن 44 بایت اول (WAV Header)
            f.seek(44)
            
            # خواندن فایل در 4 تکه مساوی
            for i in range(4):
                raw_bytes = f.read(CHUNK_BYTES)
                
                # اگر فایل کوتاه بود و داده‌ای نداشت، ویژگی‌های صفر اضافه کن
                if not raw_bytes or len(raw_bytes) < 2:
                    features.extend([0, 0])
                    continue
                
                # تبدیل بایت‌های خام به آرایه‌ای از اعداد صحیح 16 بیتی (h)
                # این کار در میکروپایتون فوق‌العاده سریع و بهینه است
                samples = array.array('h', raw_bytes)
                
                # محاسبه انرژی (Energy)
                energy = sum(abs(x) for x in samples) // len(samples)
                
                # محاسبه نرخ عبور از صفر (ZCR)
                zcr = sum((samples[j] > 0) != (samples[j-1] > 0) for j in range(1, len(samples)))
                
                # اضافه کردن به لیست ویژگی‌ها
                features.extend([energy, zcr])
                
        return features
        
    except OSError:
        print(f"Error: File '{filename}' not found on Pyboard!")
        return None

# ==========================================
# اجرای برنامه اصلی
# ==========================================
# نام فایل صوتی خود را اینجا قرار دهید
TARGET_FILE = "on_1.wav" 

print("===================================")
print("TinyML Audio Inference from WAV")
print("===================================\n")

start_time = time.ticks_us()

# 1. خواندن فایل و استخراج ویژگی‌ها
extracted_features = process_wav_file(TARGET_FILE)

if extracted_features:
    # 2. ارسال به مدل برای تشخیص
    predicted_word = predict_speech(extracted_features)
    
    end_time = time.ticks_us()
    duration_ms = time.ticks_diff(end_time, start_time) / 1000
    
    print(f"Features extracted: {extracted_features}")
    print(f"🎯 Predicted Word : [ {predicted_word} ]")
    print(f"⏱️ Total Time     : {duration_ms} ms (Reading + Processing)")
    
print("\nDone.")
